# WATERWATCH AI 💧🤖
## Phase 1 & Phase 3: Dataset Inspection & Exploratory Data Analysis (EDA)
**Role:** Member 1 – Data & ML Engineer  
**Project:** Explainable AI-Based Water Potability Prediction and Community Risk Monitoring System  

---

### Project Overview
The **WATERWATCH AI** system predicts water potability using physicochemical water-quality parameters. This notebook performs **Phase 1 (Dataset Collection and Inspection)** and **Phase 3 (Exploratory Data Analysis)** on the Kaggle Water Potability dataset.

### Dataset Description (Report Summary)
- **Observations:** 3,276 water samples
- **Input Features (9):** `ph`, `Hardness`, `Solids`, `Chloramines`, `Sulfate`, `Conductivity`, `Organic_carbon`, `Trihalomethanes`, `Turbidity` (all continuous `float64`)
- **Target Label (1):** `Potability` (`0 = Non-Potable`, `1 = Potable`)
- **Missing Values:** Localized to 3 features: `Sulfate` (23.84%), `ph` (14.99%), and `Trihalomethanes` (4.95%)
- **Class Balance:** 1,998 Non-Potable (60.99%) vs. 1,278 Potable (39.01%) — mild class imbalance

### 1. Library Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set aesthetic visual defaults
sns.set_theme(style='whitegrid', font_scale=1.0)
plt.rcParams['figure.dpi'] = 120

# Load dataset (relative path)
import os
# Robust dataset path: checks data/raw/ before falling back
DATA_PATH = '../data/raw/water_potability.csv' if os.path.exists('../data/raw/water_potability.csv') else ('data/raw/water_potability.csv' if os.path.exists('data/raw/water_potability.csv') else '../data/water_potability.csv')
df = pd.read_csv(DATA_PATH)
print(f"Dataset successfully loaded from: {DATA_PATH}")

# First 5 and Last 5 Records
print("=== FIRST FIVE RECORDS ===")
display(df.head())

print("\n=== LAST FIVE RECORDS ===")
display(df.tail())

### 2. Dataset Dimensions, Schema & Data Types

In [ ]:
print(f"Total Rows:    {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}\n")

print("--- Column Data Types & Non-Null Counts ---")
df.info()

### 3. Missing Value Analysis & Duplicate Audit

In [ ]:
# Missing value counts and percentages
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Percentage (%)': missing_pct.round(2)
})

print("=== MISSING VALUE SUMMARY ===")
display(missing_df[missing_df['Missing Count'] > 0])
print(f"Total Missing Cells Across Dataset: {df.isnull().sum().sum()}")

# Duplicate row check
dups = df.duplicated().sum()
print(f"\nTotal Duplicate Records: {dups}")
assert dups == 0, "Warning: Duplicate rows detected!"

### 4. 5-Number Summary & Target Class Distribution

In [ ]:
# 5-Number Statistical Summary
print("=== STATISTICAL SUMMARY (5-Number Summary + Mean & Std) ===")
display(df.describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(2))

# Target Class Distribution
print("\n=== TARGET CLASS DISTRIBUTION ('Potability') ===")
class_counts = df['Potability'].value_counts()
class_pcts = (df['Potability'].value_counts(normalize=True) * 100).round(2)

class_dist = pd.DataFrame({
    'Count': class_counts,
    'Percentage (%)': class_pcts
}, index=[0, 1])
class_dist.index.name = 'Potability Class'
display(class_dist)

---
## Phase 3: Exploratory Data Analysis (6 Core Visualizations)

### Visualization 1: Target Class Distribution
- **Objective:** Evaluate class balance between potable (1) and non-potable (0) samples.
- **Interpretation:** The dataset contains 1,998 non-potable records (60.99%) and 1,278 potable records (39.01%).
- **Key Observation:** Mild class imbalance exists (~61:39). Stratified splitting is required to prevent sampling bias.
- **Conclusion:** Standard accuracy is misleading; evaluation must prioritize Precision, Recall, F1-Score, and ROC-AUC.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
bars = ax.bar(['Non-Potable (0)', 'Potable (1)'], class_counts, color=['#e74c3c', '#2ecc71'], width=0.45, edgecolor='black')

for bar, count, pct in zip(bars, class_counts, class_pcts):
    height = bar.get_height()
    ax.annotate(f'{count:,} ({pct:.1f}%)',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5), textcoords='offset points',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Target Class Distribution (Potability)', fontsize=12, fontweight='bold', pad=12)
ax.set_ylabel('Number of Samples')
ax.set_ylim(0, 2300)
plt.tight_layout()
plt.show()

### Visualization 2: Feature Distribution Histograms
- **Objective:** Inspect distribution shape, central tendency, and skewness across all 9 input features.
- **Interpretation:** 8 of the 9 features exhibit roughly normal (Gaussian) bell curves. `Solids` shows moderate positive skew.
- **Key Observation:** Gaussian distribution holds well for continuous variables, supporting Gaussian Naive Bayes modeling assumptions.
- **Conclusion:** Feature values span disparate magnitudes (e.g., `Solids` in tens of thousands vs. `ph` 0-14), mandating feature scaling.

In [ ]:
features = df.columns.drop('Potability')
fig, axes = plt.subplots(3, 3, figsize=(14, 11))
axes = axes.flatten()

for i, col in enumerate(features):
    sns.histplot(df[col], kde=True, ax=axes[i], color='#3498db', edgecolor='white', bins=25)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequency')

plt.suptitle('Physicochemical Feature Distributions (Histograms with KDE)', fontsize=14, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()

### Visualization 3: Boxplots for Numerical Features (Outlier Investigation)
- **Objective:** Visualize feature dispersion, IQR spread, and identify extreme observations.
- **Interpretation:** Features are standardized to zero mean and unit variance ($Z$-scores) to compare spreads simultaneously.
- **Key Observation:** Outliers exist symmetrically beyond the $1.5 \times IQR$ threshold for all features, but outlier rates are low (0.34% to 2.53%).
- **Conclusion:** Statistical outliers identified by the 1.5 x IQR rule fall within plausible chemical limits for untreated water sources. Because severe contamination events naturally produce extreme concentrations, these observations are retained to preserve critical screening signals rather than discarded as measurement noise.

In [ ]:
# Z-score standardization for comparative visual plotting
norm_df = (df[features] - df[features].mean()) / df[features].std()

fig, ax = plt.subplots(figsize=(13, 6))
sns.boxplot(data=norm_df, ax=ax, palette='Blues_r', flierprops=dict(marker='o', markersize=3, alpha=0.5))
ax.set_title('Standardized Feature Boxplots (Outlier & Dispersion Comparison)', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Standardized Scale (Z-Score)')
ax.set_xticks(range(len(features)))
ax.set_xticklabels(features, rotation=30, ha='right', fontsize=10)
plt.tight_layout()
plt.show()

### Visualization 4: Feature Correlation Matrix Heatmap
- **Objective:** Investigate collinearity among features and linear correlation with target `Potability`.
- **Interpretation:** Pearson correlation between all predictors and `Potability` is virtually zero ($-0.030 \le r \le +0.034$).
- **Key Observation:** No single chemical parameter linearly distinguishes potable from non-potable water.
- **Conclusion:** Negligible linear correlation coefficients (|r| <= 0.034 with Potability) confirm that individual features possess virtually zero bivariate linear separating power. Effective prediction requires multi-variable non-linear algorithms (such as Decision Trees) or joint probabilistic classifiers (such as Gaussian Naive Bayes) that capture multidimensional feature interactions.

In [ ]:
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='coolwarm', vmin=-0.1, vmax=0.1, center=0,
            square=True, linewidths=0.5, cbar_kws={'label': 'Pearson Correlation Coefficient'}, ax=ax)
ax.set_title('Feature Correlation Matrix (Triangular Heatmap)', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

### Visualization 5: Feature-Wise Distributions Conditioned on Potability
- **Objective:** Compare probability density functions $P(X_i | \text{Potable})$ vs. $P(X_i | \text{Non-Potable})$.
- **Interpretation:** The density curves for Potable (green) and Non-Potable (red) almost completely overlap for every individual parameter.
- **Key Observation:** Univariate separation is impossible. Potability is determined by multi-parameter chemical interactions.
- **Conclusion:** Demonstrates the explicit need for machine learning to detect high-dimensional non-linear boundaries.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 11))
axes = axes.flatten()

for i, col in enumerate(features):
    sns.kdeplot(data=df, x=col, hue='Potability', common_norm=False,
                palette={0: '#e74c3c', 1: '#2ecc71'}, ax=axes[i], fill=True, alpha=0.25)
    axes[i].set_title(f'{col} by Potability', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Density')

plt.suptitle('Feature Probability Densities Conditioned on Potability (0 vs 1)', fontsize=14, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()

### Visualization 6: Missing-Value Visualization Matrix
- **Objective:** Examine the pattern, distribution, and co-occurrence of missing entries across samples.
- **Interpretation:** Missing values are isolated to `Sulfate`, `ph`, and `Trihalomethanes`, randomly dispersed across records.
- **Key Observation:** Missing values are localized to Sulfate (23.84%), pH (14.99%), and Trihalomethanes (4.95%) without obvious structural block-dropout across records. While the true missingness mechanism cannot be formally proven without external audit logs, median imputation provides a conservative, robust strategy that preserves complete sample size without dropping over 39% of the dataset.
- **Conclusion:** Confirms our decision to apply median imputation rather than record deletion.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='binary_r', ax=ax)
ax.set_title('Missing Value Matrix (White Stripes Indicate Missing Cells)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Dataset Features')
ax.set_xticks(np.arange(len(df.columns)) + 0.5)
ax.set_xticklabels(df.columns, rotation=30, ha='right', fontsize=10)
plt.tight_layout()
plt.show()